In [1]:
# csherwood@usgs.gov, 2026-08-20, generated with Claude Sonnet 5
#
# Diagnostic notebook: xr.open_dataset(his_url) fails on the new env but
# the file is visible on the THREDDS server, so this isolates whether the
# problem is the OPeNDAP/netCDF stack (engine, netCDF4/h5netcdf/pydap,
# libnetcdf, curl/libcurl SSL) rather than the server itself. Run top to
# bottom; each section is independent so you can stop at the first failure.

In [2]:
his_url = (
    "https://geoport.whoi.edu/thredds/dodsC/"
    "vortexfs1/usgs/Projects/Helene2024/helene77/Output_89pct/"
    "coawst_gomsab_his.nc"
)

In [3]:
# 0. Record the conda ssl_verify setting for this run -- IT support had us
# switch from conda's bundled cert store to the Windows trust store
# (`conda config --set ssl_verify truststore`, user-level .condarc, so it
# applies across all envs including this one). Logging it here so this
# rerun is self-documenting.
import subprocess
print(subprocess.run(["conda", "config", "--show", "ssl_verify"],
                      capture_output=True, text=True, shell=True).stdout)

ssl_verify: truststore



In [4]:
# 1. Package versions -- compare this printout against the old env if you
# ever find a record of it (pip freeze, conda list, environment.yml)
import xarray, netCDF4, importlib

print("xarray  ", xarray.__version__)
print("netCDF4 ", netCDF4.__version__)
print("HDF5 lib", netCDF4.__hdf5libversion__)
print("netcdf-c lib", netCDF4.__netcdf4libversion__)

for pkg in ["h5netcdf", "pydap", "cftime", "dask"]:
    try:
        m = importlib.import_module(pkg)
        print(pkg.ljust(10), getattr(m, "__version__", "unknown"))
    except ImportError:
        print(pkg.ljust(10), "NOT INSTALLED")

xarray   2026.7.0
netCDF4  1.7.4
HDF5 lib 2.1.0
netcdf-c lib 4.10.1
h5netcdf   1.8.1
pydap      3.5.9
cftime     1.6.5
dask       2026.7.1


In [5]:
# 2. Raw DAP access, bypassing xarray entirely -- if this fails too, the
# problem is below xarray (netCDF4/libnetcdf/curl), not xarray itself
import netCDF4 as nc4

try:
    ds_raw = nc4.Dataset(his_url)
    print("netCDF4.Dataset OPENED OK")
    print("variables:", list(ds_raw.variables)[:10], "...")
    ds_raw.close()
except Exception as e:
    print("netCDF4.Dataset FAILED:", repr(e))

netCDF4.Dataset OPENED OK
variables: ['ntimes', 'ndtfast', 'dt', 'dtfast', 'dstart', 'nHIS', 'ndefHIS', 'nRST', 'Falpha', 'Fbeta'] ...


In [6]:
# 3. xr.open_dataset with each backend engine explicitly, so you can see
# which engine(s) fail and with what error -- default engine selection can
# silently differ between environments depending on what's installed
import xarray as xr

for engine in ["netcdf4", "h5netcdf", "pydap"]:
    try:
        ds = xr.open_dataset(his_url, engine=engine)
        print(f"{engine:10s} OK  -- {len(ds.data_vars)} vars, dims={dict(ds.dims)}")
        ds.close()
    except Exception as e:
        print(f"{engine:10s} FAILED -- {type(e).__name__}: {e}")

netcdf4    OK  -- 121 vars, dims={'tracer': 2, 's_rho': 16, 's_w': 17, 'eta_rho': 512, 'xi_rho': 833, 'eta_u': 512, 'xi_u': 832, 'eta_v': 511, 'xi_v': 833, 'eta_psi': 511, 'xi_psi': 832, 'ocean_time': 121}


C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_14124\3125488396.py:9: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"{engine:10s} OK  -- {len(ds.data_vars)} vars, dims={dict(ds.dims)}")


h5netcdf   FAILED -- FileNotFoundError: https://geoport.whoi.edu/thredds/dodsC/vortexfs1/usgs/Projects/Helene2024/helene77/Output_89pct/coawst_gomsab_his.nc


C:\Users\csherwood\AppData\Local\miniforge3\envs\CRS11\Lib\site-packages\pydap\handlers\dap.py:184: UserWarning: PyDAP was unable to determine the DAP protocol defaulting to DAP2. DAP2 is consider legacy and may result in slower responses. 
Consider replacing `http` in your `url` with either `dap2` or `dap4` to specify the DAP protocol (e.g. `dap2://<data_url>` or `dap4://<data_url>`).  For more 
information, go to https://www.opendap.org/faq-page.
  warnings.warn(


pydap      OK  -- 121 vars, dims={'tracer': 2, 's_rho': 16, 's_w': 17, 'eta_rho': 512, 'xi_rho': 833, 'eta_u': 512, 'xi_u': 832, 'eta_v': 511, 'xi_v': 833, 'eta_psi': 511, 'xi_psi': 832, 'ocean_time': 121}


C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_14124\3125488396.py:9: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"{engine:10s} OK  -- {len(ds.data_vars)} vars, dims={dict(ds.dims)}")


In [7]:
# 4. Plain HTTPS reachability + SSL, no netCDF stack involved -- rules out
# certificate/proxy/firewall issues introduced by the new env
import requests

das_url = his_url + ".das"  # OPeNDAP metadata endpoint, small and fast
try:
    r = requests.get(das_url, timeout=15)
    print("status:", r.status_code)
    print(r.text[:300])
except Exception as e:
    print("requests FAILED:", repr(e))

status: 200
Attributes {
    ntimes {
        String long_name "number of long time-steps";
    }
    ndtfast {
        String long_name "number of short time-steps";
    }
    dt {
        String long_name "size of long time-steps";
        String units "second";
    }
    dtfast {
        String long_name "si


In [8]:
# 5. If cell 3 shows netcdf4/h5netcdf failing but pydap working (or vice
# versa), that pins the problem to a specific backend -- worth an explicit
# full traceback on whichever engine matches what the main notebook uses
# (xr.open_dataset(his_url) with no engine= argument, i.e. netCDF4 default)
ds_his = xr.open_dataset(his_url)  # let it raise with full traceback

In [9]:
# 6. Check for a dangling CA-bundle env var. netcdf-c's internal DAP/curl
# client often honors these; if either points at a path that no longer
# exists (e.g. the OLD miniforge env you said was deleted), curl fails
# SSL verification with a generic "I/O failure" -- exactly errno -68.
# requests (cell 4) usually falls back to its own certifi bundle, so it
# can succeed even when this is broken, which is consistent with a DAP
# client failure that isn't a plain network/reachability problem.
import os

for var in ["SSL_CERT_FILE", "CURL_CA_BUNDLE", "REQUESTS_CA_BUNDLE", "SSL_CERT_DIR"]:
    val = os.environ.get(var)
    if val is None:
        print(f"{var:18s} not set")
    else:
        print(f"{var:18s} = {val}   (exists: {os.path.exists(val)})")

SSL_CERT_FILE      = C:\Users\csherwood\AppData\Local\miniforge3\envs\CRS11\Library\ssl\cacert.pem   (exists: True)
CURL_CA_BUNDLE     not set
REQUESTS_CA_BUNDLE not set
SSL_CERT_DIR       = C:\Users\csherwood\AppData\Local\miniforge3\envs\CRS11\Library\ssl\certs   (exists: True)


In [10]:
# 7. pydap (already installed) is a pure-Python OPeNDAP client -- it does
# NOT go through netcdf-c's internal libcurl DAP code. Confirmed working
# previously; rerunning here alongside the netcdf4-engine retest above so
# both results are from the same ssl_verify setting for comparison.
import xarray as xr
try:
    ds = xr.open_dataset(his_url, engine="pydap")
    print("pydap OK --", len(ds.data_vars), "vars, dims=", dict(ds.dims))
except Exception as e:
    print("pydap FAILED:", type(e).__name__, e)

C:\Users\csherwood\AppData\Local\miniforge3\envs\CRS11\Lib\site-packages\pydap\handlers\dap.py:184: UserWarning: PyDAP was unable to determine the DAP protocol defaulting to DAP2. DAP2 is consider legacy and may result in slower responses. 
Consider replacing `http` in your `url` with either `dap2` or `dap4` to specify the DAP protocol (e.g. `dap2://<data_url>` or `dap4://<data_url>`).  For more 
information, go to https://www.opendap.org/faq-page.
  warnings.warn(


pydap OK -- 121 vars, dims= {'tracer': 2, 's_rho': 16, 's_w': 17, 'eta_rho': 512, 'xi_rho': 833, 'eta_u': 512, 'xi_u': 832, 'eta_v': 511, 'xi_v': 833, 'eta_psi': 511, 'xi_psi': 832, 'ocean_time': 121}


C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_14124\2431607118.py:8: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("pydap OK --", len(ds.data_vars), "vars, dims=", dict(ds.dims))


In [11]:
# %pip install cryptography -q
import ssl, socket
from cryptography import x509
from cryptography.hazmat.backends import default_backend

host = "geoport.whoi.edu"
ctx = ssl._create_unverified_context()
with socket.create_connection((host, 443), timeout=15) as sock:
    with ctx.wrap_socket(sock, server_hostname=host) as s:
        der = s.getpeercert(binary_form=True)

cert = x509.load_der_x509_certificate(der, default_backend())
print("Subject:", cert.subject.rfc4514_string())
print("Issuer :", cert.issuer.rfc4514_string())
print("Self-signed (subject == issuer):", cert.subject == cert.issuer)

Subject: CN=gravel.whoi.edu
Issuer : CN=YE2,O=Let's Encrypt,C=US
Self-signed (subject == issuer): False


In [12]:
# %pip install truststore -q
import truststore
truststore.inject_into_ssl()

import requests
das_url = his_url + ".das"
try:
    r = requests.get(das_url, timeout=15)
    print("status:", r.status_code)
    print(r.text[:300])
except Exception as e:
    print("requests FAILED:", repr(e))

status: 200
Attributes {
    ntimes {
        String long_name "number of long time-steps";
    }
    ndtfast {
        String long_name "number of short time-steps";
    }
    dt {
        String long_name "size of long time-steps";
        String units "second";
    }
    dtfast {
        String long_name "si


In [13]:
import os
os.environ["CURL_SSL_BACKEND"] = "schannel"

import xarray as xr
try:
    ds = xr.open_dataset(his_url)
    print("netcdf4 engine OK with schannel --", len(ds.data_vars), "vars")
except Exception as e:
    print("netcdf4 engine still FAILED:", type(e).__name__, e)

netcdf4 engine OK with schannel -- 121 vars
